# Lab - Chức năng Softmax
Trong Lab này, chúng ta sẽ khám phá hàm softmax. Hàm này được sử dụng trong cả Hồi quy Softmax và trong Mạng thần kinh khi giải quyết các vấn đề Phân loại nhiều lớp.  

<center> <img  src="./images/C2_W2_Softmax_Header.PNG" width="600" /> <center/>


In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from IPython.display import display, Markdown, Latex
from sklearn.datasets import make_blobs
%matplotlib widget
from matplotlib.widgets import Slider
from lab_utils_common import dlc
from lab_utils_softmax import plt_softmax
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

> **Lưu ý**: Thông thường, trong khóa học này, sổ ghi chép sử dụng quy ước đếm bắt đầu bằng 0 và kết thúc bằng N-1, $\sum_{i=0}^{N-1}$, trong khi các bài giảng bắt đầu bằng 1 và kết thúc bằng N, $\sum_{i=1}^{N}$. Điều này là do mã thường bắt đầu lặp lại bằng 0 trong khi giảng, việc đếm từ 1 đến N dẫn đến các phương trình gọn gàng hơn, rõ ràng hơn. Sổ ghi chép này có nhiều phương trình hơn mức thông thường của Lab và do đó sẽ phá vỡ quy ước và sẽ đếm từ 1 đến N.


## Chức năng Softmax
Trong cả hồi quy softmax và neural network với đầu ra Softmax, N đầu ra được tạo và một đầu ra được chọn làm danh mục dự đoán. Trong cả hai trường hợp, vectơ $\mathbf{z}$ được tạo bởi hàm tuyến tính được áp dụng cho hàm softmax. Hàm softmax chuyển đổi $\mathbf{z}$ thành phân bố xác suất như mô tả bên dưới. Sau khi áp dụng softmax, mỗi đầu ra sẽ nằm trong khoảng từ 0 đến 1 và các đầu ra sẽ cộng thành 1, do đó chúng có thể được hiểu là xác suất. Đầu vào lớn hơn sẽ tương ứng với xác suất đầu ra lớn hơn.
<center> <img  src="./images/C2_W2_SoftmaxReg_NN.png" width="600" />


Hàm softmax có thể được viết:
$$a_j = \frac{e^{z_j}}{ \sum_{k=1}^{N}{e^{z_k} }} \tag{1}$$
Đầu ra $\mathbf{a}$ là một vectơ có độ dài N, vì vậy đối với hồi quy softmax, bạn cũng có thể viết:
\bắt đầu{căn chỉnh}
\mathbf{a}(x) =
\bắt đầu{bmatrix}
P(y = 1 | \mathbf{x}; \mathbf{w},b) \\
\vdots \\
P(y = N | \mathbf{x}; \mathbf{w},b)
\end{bmatrix}
=
\frac{1}{ \sum_{k=1}^{N}{e^{z_k} }}
\bắt đầu{bmatrix}
e^{z_1} \\
\vdots \\
e^{z_{N}} \\
\end{bmatrix} \tag{2}
\end{căn chỉnh}


Điều này cho thấy đầu ra là một vectơ xác suất. Mục nhập đầu tiên là xác suất đầu vào là danh mục đầu tiên với đầu vào $\mathbf{x}$ và các tham số $\mathbf{w}$ và $\mathbf{b}$.  
Hãy tạo một triển khai NumPy:


In [2]:
def my_softmax(z):
    ez = np.exp(z)              #element-wise exponenial
    sm = ez/np.sum(ez)
    return(sm)

Bên dưới, hãy thay đổi giá trị của đầu vào `z` bằng cách sử dụng thanh trượt.


In [3]:
plt.close("all")
plt_softmax(my_softmax)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Khi bạn thay đổi các giá trị của z ở trên, có một số điều cần lưu ý:
* số mũ trong tử số của softmax phóng to những khác biệt nhỏ trong các giá trị 
* tổng các giá trị đầu ra thành một
* softmax bao trùm tất cả các đầu ra. Ví dụ: thay đổi trong `z0` sẽ thay đổi giá trị của `a0`-`a3`. So sánh điều này với các kích hoạt khác như ReLU hoặc Sigmoid có một đầu vào và một đầu ra.


## cost
<center> <img  src="./images/C2_W2_SoftMaxCost.png" width="400" /> <center/>


Hàm mất mát liên quan đến Softmax, mất mát entropy chéo, là:
\bắt đầu{phương trình}
  L(\mathbf{a},y)=\begin{case}
    -log(a_1), & \text{if $y=1$}.\\
        &\vdots\\
     -log(a_N), & \text{if $y=N$}
  \end{case} \tag{3}
\end{phương trình}

Trong đó y là danh mục target cho ví dụ này và $\mathbf{a}$ là đầu ra của hàm softmax. Cụ thể, các giá trị trong $\mathbf{a}$ là các xác suất có tổng bằng một.
>**Nhớ lại:** Trong khóa học này, Mất mát là một ví dụ trong khi cost bao gồm tất cả các ví dụ. 
 
 
Lưu ý ở (3) ở trên, chỉ có dòng tương ứng với target mới góp phần thua lỗ, các dòng khác bằng 0. Để viết Hàm cost, chúng ta cần một 'hàm chỉ báo' sẽ bằng 1 khi chỉ số phù hợp với target và bằng 0 nếu ngược lại. 
    $$\mathbf{1}\{y == n\} = =\begin{cases}
    1, & \text{if $y==n$}.\\
    0, & \text{otherwise}.
  \end{cases}$$
Bây giờ cost là:
\bắt đầu{căn chỉnh}
J(\mathbf{w},b) = -\frac{1}{m} \left[ \sum_{i=1}^{m} \sum_{j=1}^{N} 1\left\{y^{(i)} == j\right\} \log \frac{e^{z^{(i)__j}}{\sum_{k=1}^N e^{z^{(i)__k} }\right] \thẻ{4}
\end{căn chỉnh}

Trong đó $m$ là số lượng ví dụ, $N$ là số lượng đầu ra. Đây là mức trung bình của tất cả các tổn thất.


## Dòng chảy căng
Lab này sẽ thảo luận về hai cách triển khai softmax, mất entropy chéo trong Tensorflow, phương pháp 'rõ ràng' và phương pháp 'ưa thích'. Cái trước là đơn giản nhất trong khi cái sau ổn định hơn về mặt số lượng.

Hãy bắt đầu bằng cách tạo một tập dữ liệu để training mô hình phân loại nhiều lớp.


In [4]:
# tạo tập dữ liệu chẳng hạncenters = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers=centers, cluster_std=1.0,random_state=30)

### Tổ chức *Rõ ràng*


Mô hình bên dưới được triển khai với softmax dưới dạng kích hoạt trong lớp Dày đặc cuối cùng.
Hàm mất được chỉ định riêng trong lệnh `compile`. 

Hàm mất mát là `SparseCategoricalCrossentropy`. Sự mất mát này được mô tả ở (3) ở trên. Trong mô hình này, softmax diễn ra ở lớp cuối cùng. Hàm mất có đầu ra softmax là vectơ xác suất.


In [5]:
model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'softmax')    # < softmax activation here
    ]
)
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

model.fit(
    X_train,y_train,
    epochs=10
)
        

Epoch 1/10
63/63 [==============================] - 0s 966us/step - loss: 0.8312
Epoch 2/10
63/63 [==============================] - 0s 1ms/step - loss: 0.3203
Epoch 3/10
63/63 [==============================] - 0s 1ms/step - loss: 0.1408
Epoch 4/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0847
Epoch 5/10
63/63 [==============================] - 0s 944us/step - loss: 0.0626
Epoch 6/10
63/63 [==============================] - 0s 974us/step - loss: 0.0515
Epoch 7/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0447
Epoch 8/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0402
Epoch 9/10
63/63 [==============================] - 0s 922us/step - loss: 0.0361
Epoch 10/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0332


Vì softmax được tích hợp vào lớp đầu ra nên đầu ra là một vectơ xác suất.


In [6]:
p_nonpreferred = model.predict(X_train)
print(p_nonpreferred [:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

[[3.39e-03 1.02e-02 9.73e-01 1.30e-02]
 [9.97e-01 3.46e-03 9.35e-06 9.02e-06]]
largest value 0.9999994 smallest value 4.1378247e-09


###Ưu tiên <img align="Right" src="./images/C2_W2_softmax_accurate.png"  style=" width:400px; padding: 10px 20px ; ">
Nhớ lại bài giảng, có thể thu được kết quả ổn định và chính xác hơn nếu kết hợp softmax và tổn thất trong quá trình training.   Điều này được kích hoạt bởi tổ chức 'ưu tiên' được hiển thị ở đây.


Trong tổ chức ưu tiên, lớp cuối cùng có kích hoạt tuyến tính. Vì lý do lịch sử, kết quả đầu ra ở dạng này được gọi là *logits*. Hàm mất có một đối số bổ sung: `from_logits = True`. Điều này thông báo cho hàm mất mát rằng thao tác softmax phải được đưa vào tính toán tổn thất. Điều này cho phép thực hiện tối ưu hóa.


In [7]:
preferred_model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'linear')   #<-- Note
    ]
)
preferred_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  #<-- Note
    optimizer=tf.keras.optimizers.Adam(0.001),
)

preferred_model.fit(
    X_train,y_train,
    epochs=10
)
        

Epoch 1/10
63/63 [==============================] - 0s 1ms/step - loss: 1.0936
Epoch 2/10
63/63 [==============================] - 0s 1ms/step - loss: 0.4933
Epoch 3/10
63/63 [==============================] - 0s 1ms/step - loss: 0.2809
Epoch 4/10
63/63 [==============================] - 0s 1ms/step - loss: 0.1506
Epoch 5/10
63/63 [==============================] - 0s 966us/step - loss: 0.0916
Epoch 6/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0694
Epoch 7/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0592
Epoch 8/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0527
Epoch 9/10
63/63 [==============================] - 0s 919us/step - loss: 0.0489
Epoch 10/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0457


#### Xử lý đầu ra
Lưu ý rằng trong mô hình ưu tiên, kết quả đầu ra không phải là xác suất mà có thể dao động từ số âm lớn đến số dương lớn. Đầu ra phải được gửi qua softmax khi thực hiện dự đoán có xác suất. 
Hãy xem các kết quả đầu ra mô hình ưa thích:


In [8]:
p_preferred = preferred_model.predict(X_train)
print(f"two example output vectors:\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

two example output vectors:
 [[-2.72 -4.45  2.9  -1.37]
 [ 6.42  1.32 -1.32 -6.74]]
largest value 12.410254 smallest value -13.030992


Các dự đoán đầu ra không phải là xác suất!
Nếu đầu ra mong muốn là xác suất thì đầu ra phải được xử lý bằng [softmax](https://www.tensorflow.org/api_docs/python/tf/nn/softmax).


In [9]:
sm_preferred = tf.nn.softmax(p_preferred).numpy()
print(f"two example output vectors:\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))

two example output vectors:
 [[3.55e-03 6.34e-04 9.82e-01 1.38e-02]
 [9.94e-01 6.05e-03 4.35e-04 1.92e-06]]
largest value 0.9999995 smallest value 1.5196424e-11


Để chọn danh mục có khả năng xảy ra cao nhất, không cần phải có softmax. Người ta có thể tìm chỉ số của đầu ra lớn nhất bằng cách sử dụng [np.argmax()](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html).


In [10]:
for i in range(5):
    print( f"{p_preferred[i]}, category: {np.argmax(p_preferred[i])}")

[-2.72 -4.45  2.9  -1.37], category: 2
[ 6.42  1.32 -1.32 -6.74], category: 0
[ 4.64  1.5  -1.31 -5.3 ], category: 0
[-0.29  3.76 -3.11 -1.58], category: 1
[-0.64 -6.05  5.23 -5.2 ], category: 2


## SparseCategorialCrossentropy hoặc CategoricalCrossEntropy
Tensorflow có hai định dạng tiềm năng cho các giá trị target và việc lựa chọn mức tổn thất sẽ xác định giá trị nào được mong đợi.
- SparseCategorialCrossentropy: mong đợi target là số nguyên tương ứng với chỉ mục. Ví dụ: nếu có 10 giá trị target tiềm năng thì y sẽ nằm trong khoảng từ 0 đến 9. 
- CategoricalCrossEntropy: Dự kiến giá trị đích của một ví dụ sẽ được mã hóa một lần trong đó giá trị tại chỉ mục đích là 1 trong khi các mục N-1 khác bằng 0. Một ví dụ với 10 giá trị target tiềm năng, trong đó target là 2 sẽ là [0,0,1,0,0,0,0,0,0,0].


## Chúc mừng!
Trong Lab này bạn 
- Trở nên quen thuộc hơn với hàm softmax và cách sử dụng nó trong hồi quy softmax cũng như trong kích hoạt softmax trong neural networks. 
- Tìm hiểu cách xây dựng mô hình ưa thích trong Tensorflow:
    - Không kích hoạt ở lớp cuối cùng (giống như kích hoạt tuyến tính)
    - Hàm mất mát chéo phân loại thưa thớt
    - sử dụng from_logits=True
- Nhận thấy rằng không giống như ReLU và Sigmoid, softmax mở rộng ra nhiều đầu ra.
